# Analysis of finished ```.npz``` files

This Jupyter-Notebook should load the ```.npz```-files which contain the background-residual-field without any coil current, the field with coil current and the simulation results for the field with coil current. *Note* that ```.npz``` files (numpy-zip) need to be within the specified directory. This Code is not supposed to by tidy, but include everything necessary for different visualizations of the result.
_______________
Created 20. May, 2026 by Gregor Bock

(0378 1735; ge27doc)

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

import Ausgelagerte_Funktionen_Versuchsauswertung as fkt

## Load Data
- Background field (and positions)
- Measured field (and positions)
- Simulated field (and positions)
- Insert zero field values of the QSpin manually!!!

<span style="color:red">The directions of the map and the zero field do not match!!!</span>
- MSR-x = QSpin-x
- MSR-y = QSpin-z
- MSR-z = QSpin-y

In [ ]:
folder_path_background = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Experiment2\background_01_2026-05-13_10-48-57\map\points"
folder_path_experiment = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Experiment2\strom_10mA_03_2026-05-13_14-44-44\map\points"
folder_path_simulation = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\B_tot_results"

# Fill in field zero values here: General structure: [x_QSpin, y_QSpin, z_QSpin]
field_zero_values_background = [-1541, -1741, 1764] * 10**(-12)        # Background -> Tag mit großer/mittlerer Spule

# field_zero_values = [-2312, -1834, 1588] * 10**(-12)      # mittlere Spule, 1mA
# field_zero_values = [-11305, -2671, 11] * 10**(-12)       # mittlere Spule, 10mA
# field_zero_values = [-6347, -2203, 867] * 10**(-12)       # mittlere Spule, 5mA
# field_zero_values = [-1849, -1749, 1689] * 10**(-12)      # mittlere Spule, 0.5mA

# field_zero_values = [1240, -1645, 1926] * 10**(-12)       # grosse Spule, 1mA
field_zero_values = [26472, 223, 5857] * 10**(-12)        # grosse Spule, 10mA
# field_zero_values = [-12479, -845, 3758] * 10**(-12)      # grosse Spule, 5mA
# field_zero_values = [-73, -1615, 1918] * 10**(-12)        # grosse Spule, 0.5mA

field_zero_values_background = [field_zero_values_background[0], field_zero_values_background[2], field_zero_values_background[1]]  # Convert the QSpin Coordinate System into the MSR-Mapper-Coordinate System (Note: This only holds, if the cable of the QSpin is at the bottom!!!)
field_zero_values = [field_zero_values[0], field_zero_values[2], field_zero_values[1]]  # Convert the QSpin Coordinate System into the MSR-Mapper-Coordinate System (Note: This only holds, if the cable of the QSpin is at the bottom!!!)

# If .npz file does not hold geometric track data, specify it here
L_x = 0.800                                         # length of the mapped volume in x-direction
L_y = 0.800                                         # length of the mapped volume in y-direction
L_z = 0.400                                         # length of the mapped volume in z-direction
step_size = 0.200                                   # size of the grid steps

shift_x = 0                                         # shift of the mapped volume in x-direction
shift_y = 0                                         # shift of the mapped volume in y-direction
shift_z = 0                                         # shift of the mapped volume in z-direction

# Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
target_point_coord_exp, B_target_point_background, B_target_point_exp = fkt.load_data_from_folder(folder_path_background, folder_path_experiment, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)


# Load data from simulation directly from .npz file (no difficult iterations necessary)
file_dir = os.path.dirname(folder_path_experiment)
parent_dir = os.path.dirname(file_dir)
filename = 'B_tot_results\\B_tot_rault_' + os.path.basename(parent_dir) + '.npz'

data_sim = np.load(filename)
# Debug
# print(f'The names of the arrays of the .npz-file are: {sorted(data_sim.files)}')

B_tot = data_sim['B_field']

target_point_coord_calc = data_sim['points']
stream_func = data_sim['stream_function']
C_matrix = data_sim['Coupling_matrix']
d_coil = data_sim['d_coil']
current = data_sim['current']
n_windings = data_sim['n_windings']
all_coils = data_sim['all_coils']

## Load additional maps

Always combination of measurement and simulation, no background.

In [ ]:
dic_of_files_and_data = {'File_Basename': ['strom_10mA_03_2026-05-13_14-44-44', 'strom_5mA_04_2026-05-13_15-39-56', 'strom_05mA_05_2026-05-13_17-07-47', 'strom_0001A_02_2026-05-13_12-39-37', 'strom_10mA_mittel_07_2026-05-13_19-24-54', 'strom_5mA_mittel_08_2026-05-13_20-28-47', 'strom_05mA_mittel_09_2026-05-13_21-24-25', 'strom_0001A_mittel_06_2026-05-13_18-22-16', 'after_degauss_with_current_2026-04-27_16-54-25'],
                         'field_zero_vals': [[26472, 223, 5857], [-12479, -845, 3758], [-73, -1615, 1918], [1240, -1645, 1926], [-11305, -2671, 11], [-6347, -2203, 867], [-1849, -1749, 1689], [-2312, -1834, 1588], [0, 0, 0]]
                         }

Backgrounds = {'File_Basename': ['after_degauss_no_current_2026-04-27_16-07-03', 'background_01_2026-05-13_10-48-57'],
               'field_zero_vals': [[-1576, -1794, 1892], [-1541, -1741, 1764]]
               }

for i, field_zero_vals in enumerate(dic_of_files_and_data['field_zero_vals']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    B_0_x_QSpin = field_zero_vals[0]
    B_0_y_QSpin = field_zero_vals[1]
    B_0_z_QSpin = field_zero_vals[2]
    dic_of_files_and_data['field_zero_vals'][i] = [B_0_x_QSpin * 10**(-12), -B_0_z_QSpin * 10**(-12), -B_0_y_QSpin * 10**(-12)]

for i, field_zero_vals in enumerate(Backgrounds['field_zero_vals']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    B_0_x_QSpin = field_zero_vals[0]
    B_0_y_QSpin = field_zero_vals[1]
    B_0_z_QSpin = field_zero_vals[2]
    dic_of_files_and_data['field_zero_vals'][i] = [B_0_x_QSpin * 10**(-12), -B_0_z_QSpin * 10**(-12), -B_0_y_QSpin * 10**(-12)]

folder_exp = "D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\Experiments\\"
folder_calc = "D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\B_tot_results\\"
praefix_calc = "B_tot_rault_" # Change this when ready
suffix = "\\map\\points"

# If .npz file does not hold geometric track data, specify it here
L_x = 0.800                                         # length of the mapped volume in x-direction
L_y = 0.800                                         # length of the mapped volume in y-direction
L_z = 0.400                                         # length of the mapped volume in z-direction
step_size = 0.200                                   # size of the grid steps

shift_x = 0                                         # shift of the mapped volume in x-direction
shift_y = 0                                         # shift of the mapped volume in y-direction
shift_z = 0                                         # shift of the mapped volume in z-direction

Background_map = 1
folder_path_background = folder_exp + Backgrounds['File_Basename'][Background_map] + suffix

print(f'\nStarting to load data for Background file: {Backgrounds["File_Basename"][Background_map]}')
target_point_coord_exp, B_target_point_background, _ = fkt.load_data_from_folder(folder_path_background, folder_path_background, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)
for comp in range(3):
    B_target_point_background[:, comp] = B_target_point_background[:, comp] - Backgrounds['field_zero_vals'][Background_map][comp]

B_target_point = {'File_Basename': [], 'B_field_exp': [], 'B_field_calc': [],'target_point_coord_calc': [], 'stream_func': [], 'd_coil': [], 'current': [], 'n_windings': [], 'all_coils': []}

for idx, Basename in enumerate(dic_of_files_and_data['File_Basename'][4:8]):
    print(f'\nStarting to load data for file: {Basename}')
    B_target_point['File_Basename'].append(Basename)

    folder_path_experiment = folder_exp + Basename + suffix
    folder_path_calc = folder_calc + praefix_calc + Basename + '.npz'
    
    # Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
    B_target_point['B_field_exp'].append(fkt.load_data_from_folder(folder_path_background, folder_path_experiment, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)[2])
    
    # Load data of calculation
    data_sim = np.load(folder_path_calc)
    B_target_point['B_field_calc'].append(data_sim['B_field'])

    for comp in range(3):
        B_target_point['B_field_exp'][idx][:, comp] = B_target_point['B_field_exp'][idx][:, comp] - dic_of_files_and_data['field_zero_vals'][idx + 4][comp]

    B_target_point['target_point_coord_calc'].append(data_sim['points'])
    B_target_point['stream_func'].append(data_sim['stream_function'])
    B_target_point['d_coil'].append(data_sim['d_coil'])
    B_target_point['current'].append(data_sim['current'])
    B_target_point['n_windings'].append(data_sim['n_windings'])
    B_target_point['all_coils'].append(data_sim['all_coils'])

## Some simple plots :)

In [ ]:
# Plot the coil layup, such that it is clear, which experiment setup was used. Additionally, print coil-diameter, currrent and windings, since they can not be determined by the plot alone

fig_coil_layup = plt.figure()
ax_coil_layup = fig_coil_layup.add_subplot(111, projection='3d')
for k in range(all_coils.shape[0]):
    ax_coil_layup.plot(all_coils[k, :, 0], all_coils[k, :, 1], all_coils[k, :, 2], color='blue')
ax_coil_layup.set_xlabel('x')
ax_coil_layup.set_ylabel('y')
ax_coil_layup.set_zlabel('z')


print(f'The coil diameter used in the simulation is ' + r"$d_{coil}$ =" + f' {d_coil}')
print(f'The coil current used in the simulation is ' + r"$I_{coil}$ =" + f' {current}')
print(f'The number of windings of each coil used in the simulation is ' + r"$n_{windings}$ =" + f' {n_windings}')

## Plot magnetic fields (norm)

This should give some intuition on how to access and work with the given arrays. Moreover a slight intuition for the field can be obtained (maybe)

In [ ]:
# Plot total field from Background (without coil currents)
fig_B_background = plt.figure()
ax_B_background = fig_B_background.add_subplot(111, projection='3d')
sc_B_background = ax_B_background.scatter(
    target_point_coord_exp[:, 0],
    target_point_coord_exp[:, 1],
    target_point_coord_exp[:, 2],
    c = np.linalg.norm(B_target_point_background, axis=1),
    s = 100,
    cmap = 'viridis',
    alpha = 0.5,
)
fig_B_background.colorbar(sc_B_background, ax=ax_B_background, label=r'$|\mathbf{B}_{\text{background}}|$')
ax_B_background.set_xlabel('x')
ax_B_background.set_ylabel('y')


# Plot total field from simulation
fig_B_tot = plt.figure()
ax_B_tot = fig_B_tot.add_subplot(111, projection='3d')
sc_B_tot = ax_B_tot.scatter(
    target_point_coord_calc[:, 0],
    target_point_coord_calc[:, 1],
    target_point_coord_calc[:, 2],
    c = np.linalg.norm(B_tot, axis=1),
    s = 100,
    cmap = 'viridis',
    alpha = 0.5,
)
fig_B_tot.colorbar(sc_B_tot, ax=ax_B_tot, label=r'$|\mathbf{B}_{\text{total}}|$')
ax_B_tot.set_xlabel('x')
ax_B_tot.set_ylabel('y')


# Plot coarsened field from simulation for better comparison
num_calc_target_points_fine = int( (len(target_point_coord_calc) + 1) **(1/3) )
coil_plane_dist_to_origin_x = 2.34/2
coil_plane_dist_to_origin_y = 2.34/2
coil_plane_dist_to_origin_z = 2.20/2
safety_distance = 0.05
B_tot_coarse = fkt.interpolate_B_on_coarse_grid(num_calc_target_points_fine, target_point_coord_exp, coil_plane_dist_to_origin_x, coil_plane_dist_to_origin_y, coil_plane_dist_to_origin_z, safety_distance, B_tot)

fig_B_tot_coarse = plt.figure()
ax_B_tot_coarse = fig_B_tot_coarse.add_subplot(111, projection='3d')
sc_B_tot_coarse = ax_B_tot_coarse.scatter(
    target_point_coord_exp[:, 0],
    target_point_coord_exp[:, 1],
    target_point_coord_exp[:, 2],
    c = np.linalg.norm(B_tot_coarse, axis=1),
    s = 100,
    cmap = 'viridis',
    alpha = 0.5,
)
fig_B_tot_coarse.colorbar(sc_B_tot_coarse, ax=ax_B_tot_coarse, label=r'$|\mathbf{B}_{\text{total}}|$')
ax_B_tot_coarse.set_xlabel('x')
ax_B_tot_coarse.set_ylabel('y')


# Plot total field from Experiment (with coil currents)
fig_B_exp = plt.figure()
ax_B_exp = fig_B_exp.add_subplot(111, projection='3d')
sc_B_exp = ax_B_exp.scatter(
    target_point_coord_exp[:, 0],
    target_point_coord_exp[:, 1],
    target_point_coord_exp[:, 2],
    c = np.linalg.norm(B_target_point_exp, axis=1),
    s = 100,
    cmap = 'viridis',
    alpha = 0.5,
)
fig_B_exp.colorbar(sc_B_exp, ax=ax_B_exp, label=r'$|\mathbf{B}_{\text{experiment}}|$')
ax_B_exp.set_xlabel('x')
ax_B_exp.set_ylabel('y')

r = []
for i in range(len(B_target_point_exp)):
    if np.all(B_target_point_exp[i]) == 0:
        r.append(target_point_coord_exp[i, :])

print(f'Note, that there were {len(r)} points in the experiment where the mapper did not record any field.\nAt the coordinates {r} the field is set to zero manually!')

## Differences of fields

Here the difference of the computed and measured field shall be shown. This should be as small as possible!

<span style="color:red">**Note** that the directions of the field until now do not have to be he same. If a ```+``` or a ```-``` is used must be evaluated manually!!!</span>

In [ ]:
B_diff = np.absolute(B_tot_coarse + B_target_point_exp)
B_diff_norm = np.linalg.norm(B_diff, axis=1)

fig_B_diff = plt.figure()
ax_B_diff = fig_B_diff.add_subplot(111, projection='3d')
sc_B_diff = ax_B_diff.scatter(
    target_point_coord_exp[:, 0],
    target_point_coord_exp[:, 1],
    target_point_coord_exp[:, 2],
    c = B_diff_norm,
    s = 100,
    cmap = 'viridis',
    alpha = 0.5,
)
fig_B_diff.colorbar(sc_B_diff, ax=ax_B_diff, label=r'$|\mathbf{B}_{\text{difference}}|$')
ax_B_diff.set_xlabel('x')
ax_B_diff.set_ylabel('y')

# Calculate statistical measures
B_diff_variance = np.var(B_diff_norm)
B_diff_std = np.sqrt(B_diff_variance)
B_diff_average = np.average(B_diff_norm)

# Create Histogramm of differences
fig_B_diff_hist = plt.figure()
ax_B_diff_hist = fig_B_diff_hist.add_subplot()
ax_B_diff_hist.hist(B_diff_norm, 32, alpha = 0.8)

counts, bins, _ = ax_B_diff_hist.hist(B_diff_norm, bins=32, alpha=0.6)

x = np.linspace(min(B_diff_norm), max(B_diff_norm), 200)
bin_width = bins[1] - bins[0]
Gauss = (1 / (B_diff_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - B_diff_average) / B_diff_std) ** 2)
Gauss_scaled = Gauss * len(B_diff_norm) * bin_width

ax_B_diff_hist.plot(x, Gauss_scaled, 'g')

ax_B_diff_hist.set_xlabel('Difference between simulated and measured B-field in T')
ax_B_diff_hist.set_ylabel('Number of points with described difference')

print(f'The average value of the difference in the norm of the B-field is {B_diff_average * 10**12:.0f} pT\nThe standard deviation of the difference in the norm of the B-field is {B_diff_std * 10**12:.0f} pT')